# Flask Stable Diffusion Inpainting + Ngrok (Kaggle)

Menjalankan web Flask inpainting di Kaggle dengan ngrok port forwarding.

**Langkah:**
1. Aktifkan **GPU** (Settings → Accelerator → GPU)
2. Daftar di [ngrok.com](https://ngrok.com) dan ambil **authtoken** dari [dashboard](https://dashboard.ngrok.com/get-started/your-authtoken)
3. Paste authtoken di cell konfigurasi di bawah
4. Run semua cell
5. Buka URL ngrok yang muncul untuk mengakses web app

## 1. Install Dependencies

In [ ]:
!pip install -q flask pyngrok diffusers transformers accelerate safetensors

## 2. Konfigurasi Ngrok

In [ ]:
# Paste authtoken dari https://dashboard.ngrok.com/get-started/your-authtoken
NGROK_AUTHTOKEN = ""  # Contoh: "2abc123xyz..."

if not NGROK_AUTHTOKEN:
    print("⚠️ Isi NGROK_AUTHTOKEN terlebih dahulu!")
else:
    print("✓ Ngrok token terkonfigurasi")

## 3. Buat File Flask App

In [ ]:
import os
os.makedirs("/kaggle/working/templates", exist_ok=True)

In [ ]:
%%writefile /kaggle/working/app.py
"""
Flask app untuk Stable Diffusion Inpainting.
Input: gambar (wajib), prompt, mask inpainting (digambar di web).
Model dari Civit AI (URL + API key).
"""

import os
import re
import io
import base64
import hashlib
import requests
from pathlib import Path
from flask import Flask, render_template, request, jsonify
from PIL import Image
import torch

_pipeline = None
_model_path = None

app = Flask(__name__, template_folder="/kaggle/working/templates")
app.config["MAX_CONTENT_LENGTH"] = 16 * 1024 * 1024
app.config["MODEL_CACHE_DIR"] = Path("/kaggle/working/models")
app.config["UPLOAD_FOLDER"] = Path("/kaggle/working/uploads")
app.config["OUTPUT_FOLDER"] = Path("/kaggle/working/outputs")

for folder in [app.config["MODEL_CACHE_DIR"], app.config["UPLOAD_FOLDER"], app.config["OUTPUT_FOLDER"]]:
    folder.mkdir(parents=True, exist_ok=True)


def get_civitai_download_url(url: str, api_key: str) -> str:
    if "/api/download/models/" in url:
        return url.strip()
    match = re.search(r"civitai\.com/models/(\d+)", url)
    if not match:
        raise ValueError("URL tidak valid")
    model_id = match.group(1)
    api_url = f"https://civitai.com/api/v1/models/{model_id}"
    headers = {"Content-Type": "application/json"}
    if api_key:
        headers["Authorization"] = f"Bearer {api_key}"
    resp = requests.get(api_url, headers=headers, timeout=30)
    resp.raise_for_status()
    data = resp.json()
    model_versions = data.get("modelVersions", [])
    if not model_versions:
        raise ValueError("Model tidak memiliki versi")
    for version in model_versions:
        files = version.get("files", [])
        for f in files:
            if f.get("primary") or f.get("metadata", {}).get("format") == "SafeTensor":
                return f.get("downloadUrl", "")
        if files:
            return files[0].get("downloadUrl", "")
    return f"https://civitai.com/api/download/models/{model_versions[0]['id']}"


def download_civitai_model(url: str, api_key: str, save_dir: Path) -> str:
    download_url = get_civitai_download_url(url, api_key)
    save_dir.mkdir(parents=True, exist_ok=True)
    if api_key:
        sep = "&" if "?" in download_url else "?"
        download_url = f"{download_url}{sep}token={api_key}"
    cache_key = hashlib.md5(url.strip().encode()).hexdigest()
    for ext in [".safetensors", ".ckpt"]:
        cached = save_dir / f"{cache_key}{ext}"
        if cached.exists():
            return str(cached)
    headers = {"Authorization": f"Bearer {api_key}"} if api_key else {}
    resp = requests.get(download_url, headers=headers, stream=True, timeout=120)
    resp.raise_for_status()
    content_disp = resp.headers.get("Content-Disposition")
    filename = "model.safetensors"
    if content_disp and "filename=" in content_disp:
        m = re.findall(r'filename[*]?=(?:UTF-8\\'\\')?["\\']?([^"\\';\\n]+)', content_disp)
        if m:
            filename = m[0].strip()
    ext = ".safetensors" if ".safetensors" in filename.lower() else ".ckpt"
    save_path = save_dir / f"{cache_key}{ext}"
    with open(save_path, "wb") as f:
        for chunk in resp.iter_content(chunk_size=8192):
            f.write(chunk)
    return str(save_path)


def load_pipeline(model_path: str):
    global _pipeline, _model_path
    if _pipeline is not None and _model_path == model_path:
        return _pipeline
    from diffusers import StableDiffusionInpaintPipeline
    _pipeline = StableDiffusionInpaintPipeline.from_single_file(
        model_path,
        torch_dtype=torch.float16 if torch.cuda.is_available() else torch.float32,
        use_safetensors=model_path.endswith(".safetensors"),
    )
    _pipeline = _pipeline.to("cuda" if torch.cuda.is_available() else "cpu")
    _model_path = model_path
    return _pipeline


@app.route("/")
def index():
    return render_template("index.html")


@app.route("/api/load-model", methods=["POST"])
def load_model():
    data = request.get_json() or {}
    url = data.get("url", "").strip()
    api_key = data.get("api_key", "").strip()
    if not url:
        return jsonify({"success": False, "error": "URL model wajib diisi"}), 400
    try:
        model_path = download_civitai_model(url, api_key, app.config["MODEL_CACHE_DIR"])
        load_pipeline(model_path)
        return jsonify({"success": True, "message": "Model berhasil dimuat"})
    except Exception as e:
        return jsonify({"success": False, "error": str(e)}), 500


@app.route("/api/inpaint", methods=["POST"])
def inpaint():
    data = request.get_json() or {}
    image_b64 = data.get("image")
    mask_b64 = data.get("mask")
    prompt = data.get("prompt", "").strip()
    negative_prompt = data.get("negative_prompt", "blurry, low quality")
    strength = float(data.get("strength", 0.75))
    guidance_scale = float(data.get("guidance_scale", 7.5))
    num_steps = int(data.get("num_steps", 50))
    seed = int(data.get("seed", 42))
    if not image_b64:
        return jsonify({"success": False, "error": "Gambar wajib diupload"}), 400
    if not mask_b64:
        return jsonify({"success": False, "error": "Mask wajib digambar"}), 400
    if not prompt:
        return jsonify({"success": False, "error": "Prompt wajib diisi"}), 400
    if _pipeline is None:
        return jsonify({"success": False, "error": "Model belum dimuat"}), 400
    try:
        if "," in image_b64:
            image_b64 = image_b64.split(",", 1)[1]
        init_image = Image.open(io.BytesIO(base64.b64decode(image_b64))).convert("RGB")
        if "," in mask_b64:
            mask_b64 = mask_b64.split(",", 1)[1]
        mask_image = Image.open(io.BytesIO(base64.b64decode(mask_b64))).convert("L")
        if init_image.size != mask_image.size:
            mask_image = mask_image.resize(init_image.size, Image.Resampling.LANCZOS)
        w, h = (init_image.size[0] // 8) * 8, (init_image.size[1] // 8) * 8
        init_image = init_image.resize((w, h), Image.Resampling.LANCZOS)
        mask_image = mask_image.resize((w, h), Image.Resampling.LANCZOS)
        pipe = load_pipeline(_model_path)
        generator = torch.Generator(device=pipe.device).manual_seed(seed)
        result = pipe(prompt=prompt, image=init_image, mask_image=mask_image, strength=strength,
                     guidance_scale=guidance_scale, num_inference_steps=num_steps,
                     negative_prompt=negative_prompt, generator=generator)
        buf = io.BytesIO()
        result.images[0].save(buf, format="PNG")
        return jsonify({"success": True, "image": f"data:image/png;base64,{base64.b64encode(buf.getvalue()).decode()}"})
    except Exception as e:
        return jsonify({"success": False, "error": str(e)}), 500

In [ ]:
# Copy template dari dataset jika ada, atau buat minimal
import shutil
import os

template_src = "/kaggle/input/stable-diffusion-flask/templates/index.html"
template_dst = "/kaggle/working/templates/index.html"

if os.path.exists(template_src):
    shutil.copy(template_src, template_dst)
    print("✓ Template dari dataset")
else:
    html = '''<!DOCTYPE html><html><head><meta charset="UTF-8"><title>SD Inpainting</title>
<style>body{font-family:sans-serif;background:#1a1a2e;color:#eee;max-width:800px;margin:20px auto;padding:20px;}
input,button{padding:10px;margin:5px;border-radius:8px;}
.btn{background:#00d9ff;color:#000;border:none;cursor:pointer;}
canvas{border:2px solid #0f3460;cursor:crosshair;}</style></head><body>
<h1>Stable Diffusion Inpainting</h1>
<p>1. Load Model: <input id="url" placeholder="Civit AI URL" style="width:300px"> <input id="key" type="password" placeholder="API Key"> <button class="btn" id="loadBtn">Load</button></p>
<p>2. Upload: <input type="file" id="imgInput" accept="image/*"></p>
<p>3. Gambar mask di area yang ingin di-inpaint (putih)</p>
<p>Brush: <input type="number" id="brush" min="5" max="200" value="30" style="width:60px"> px <button class="btn" id="clearBtn">Hapus Mask</button></p>
<p>Strength: <input type="number" id="strength" min="0.1" max="1" step="0.05" value="0.75" style="width:60px"> Steps: <input type="number" id="numSteps" min="10" max="150" value="50" style="width:60px"> Seed: <input type="number" id="seed" value="42" style="width:60px"></p>
<div style="position:relative;display:inline-block"><canvas id="imgC"></canvas><canvas id="maskC" style="position:absolute;left:0;top:0;pointer-events:auto"></canvas></div>
<p>4. Prompt: <input id="prompt" placeholder="deskripsi" style="width:400px"></p>
<p><button class="btn" id="runBtn">Jalankan Inpainting</button></p>
<div id="status"></div><img id="result" style="max-width:100%;margin-top:20px">
<script>
let img=null,drawing=false;
document.getElementById("loadBtn").onclick=async function(){const r=await fetch("/api/load-model",{method:"POST",headers:{"Content-Type":"application/json"},body:JSON.stringify({url:document.getElementById("url").value.trim(),api_key:document.getElementById("key").value})});const d=await r.json();document.getElementById("status").innerHTML=d.success?"<span style=color:green>"+d.message+"</span>":"<span style=color:red>"+d.error+"</span>";};
document.getElementById("imgInput").onchange=function(e){const f=e.target.files[0];if(!f)return;const i=new Image();i.onload=function(){img=i;let w=Math.floor(i.width/8)*8,h=Math.floor(i.height/8)*8;document.getElementById("imgC").width=document.getElementById("maskC").width=w;document.getElementById("imgC").height=document.getElementById("maskC").height=h;document.getElementById("imgC").getContext("2d").drawImage(i,0,0,w,h);document.getElementById("maskC").getContext("2d").clearRect(0,0,w,h);document.getElementById("imgC").parentNode.style.position="relative";document.getElementById("imgC").parentNode.style.display="inline-block";};i.src=URL.createObjectURL(f);};
document.getElementById("maskC").onmousedown=function(){drawing=true;};
document.getElementById("maskC").onmousemove=function(e){if(!drawing)return;const c=document.getElementById("maskC"),r=c.getBoundingClientRect(),x=(e.clientX-r.left)*c.width/r.width,y=(e.clientY-r.top)*c.height/r.height,ctx=c.getContext("2d"),s=document.getElementById("brush").value/2;ctx.fillStyle="rgba(255,255,255,0.7)";ctx.beginPath();ctx.arc(x,y,s,0,Math.PI*2);ctx.fill();};
document.getElementById("maskC").onmouseup=document.getElementById("maskC").onmouseleave=function(){drawing=false;};
document.getElementById("clearBtn").onclick=function(){document.getElementById("maskC").getContext("2d").clearRect(0,0,document.getElementById("maskC").width,document.getElementById("maskC").height);};
function getMask(){const c=document.createElement("canvas"),m=document.getElementById("maskC"),d=m.getContext("2d").getImageData(0,0,m.width,m.height),o=m.getContext("2d").createImageData(m.width,m.height);for(let i=0;i<d.data.length;i+=4){const v=(d.data[i]>50||d.data[i+3]>50)?255:0;o.data[i]=o.data[i+1]=o.data[i+2]=v;o.data[i+3]=255;}c.width=m.width;c.height=m.height;c.getContext("2d").putImageData(o,0,0);return c.toDataURL("image/png");}
document.getElementById("runBtn").onclick=async function(){const p=document.getElementById("prompt").value.trim();if(!p){document.getElementById("status").innerHTML="<span style=color:red>Prompt wajib</span>";return;}if(!img){document.getElementById("status").innerHTML="<span style=color:red>Upload gambar dulu</span>";return;}document.getElementById("status").innerHTML="Processing...";const r=await fetch("/api/inpaint",{method:"POST",headers:{"Content-Type":"application/json"},body:JSON.stringify({image:document.getElementById("imgC").toDataURL("image/png"),mask:getMask(),prompt:p,negative_prompt:"blurry",strength:parseFloat(document.getElementById("strength").value)||0.75,num_steps:parseInt(document.getElementById("numSteps").value)||50,seed:parseInt(document.getElementById("seed").value)||42})});const d=await r.json();if(d.success){document.getElementById("result").src=d.image;document.getElementById("status").innerHTML="<span style=color:green>Selesai!</span>";}else{document.getElementById("status").innerHTML="<span style=color:red>"+d.error+"</span>";}};
</script></body></html>'''
    with open(template_dst, 'w') as f:
        f.write(html)
    print("✓ Template dibuat (inline)")

## 4. Jalankan Flask + Ngrok

In [ ]:
import threading
import time
import sys
from pyngrok import ngrok

PORT = 5000

if not NGROK_AUTHTOKEN:
    raise ValueError("Isi NGROK_AUTHTOKEN di cell konfigurasi!")

# 1. Start Flask di thread
sys.path.insert(0, "/kaggle/working")
from app import app

def run_flask():
    app.run(host="0.0.0.0", port=PORT, debug=False, use_reloader=False)

thread = threading.Thread(target=run_flask, daemon=True)
thread.start()
time.sleep(2)  # Tunggu Flask siap

# 2. Start ngrok tunnel
ngrok.set_auth_token(NGROK_AUTHTOKEN)
public_url = ngrok.connect(PORT).public_url

print("=" * 60)
print("🌐 BUKA URL INI DI BROWSER:")
print(public_url)
print("=" * 60)
print("\nFlask + ngrok aktif. Jangan tutup notebook!")

In [ ]:
# Cell ini bisa di-skip. Flask dan ngrok sudah berjalan dari cell sebelumnya.
# Untuk menjaga session tetap hidup, jalankan cell di bawah (opsional):
import time
print("Notebook tetap berjalan. Tekan Stop untuk menghentikan server.")
try:
    while True:
        time.sleep(60)
except KeyboardInterrupt:
    pass